# Understanding Context Precision and Context Recall using Four Retrieval Scenarios

## Introduction

When evaluating a Retrieval-Augmented Generation (RAG) system, it is important to evaluate the **retrieval component** separately from the **answer generation component**.

A retriever has two primary responsibilities:

1. Retrieve information that is relevant to the user's question.
2. Retrieve all information necessary to answer the question completely.

RAGAS provides two metrics that help evaluate these responsibilities:

### Context Precision

Context Precision answers:

> How much of the retrieved information is actually useful for answering the question?

A high precision score indicates that most retrieved chunks are relevant.

A low precision score indicates that the retriever returned a significant amount of irrelevant information.

---

### Context Recall

Context Recall answers:

> Did the retriever retrieve all information necessary to answer the question?

A high recall score indicates that all required information was retrieved.

A low recall score indicates that important information was missing from the retrieved context.

---

## Example Question Used Throughout

**Question**

```text
What is Kubernetes?
```

**Reference Answer**

```text
Kubernetes is an open-source container orchestration platform
that automates deployment, scaling, and management of
containerized applications.
```

The reference answer contains several important facts:

```text
Fact 1: Open-source
Fact 2: Container orchestration platform
Fact 3: Deployment
Fact 4: Scaling
Fact 5: Management
```

These facts will be used by RAGAS when evaluating Context Recall.

---

# Scenario A: Good Retrieval

## Retrieved Context

```text
Kubernetes is an open-source container orchestration platform.

It automates deployment, scaling, and management of
containerized applications.
```

## Analysis

The retriever returned:

```text
✓ Open-source
✓ Container orchestration
✓ Deployment
✓ Scaling
✓ Management
```

No important information is missing.

No irrelevant information is present.

### Context Precision

Every retrieved chunk contributes directly to answering the question.

```text
Relevant Chunks: 100%
Irrelevant Chunks: 0%
```

Therefore:

```text
Context Precision → High
```

### Context Recall

All facts from the reference answer are present.

```text
✓ Open-source
✓ Container orchestration
✓ Deployment
✓ Scaling
✓ Management
```

Nothing is missing.

Therefore:

```text
Context Recall → High
```

### Expected Outcome

| Metric | Expected |
|----------|----------|
| Context Precision | High |
| Context Recall | High |

---

# Scenario B: Wrong Retrieval

## Retrieved Context

```text
Tomatoes grow well in summer.

Bananas are rich in potassium.
```

## Analysis

The retriever returned information unrelated to Kubernetes.

Retrieved facts:

```text
✗ Open-source
✗ Container orchestration
✗ Deployment
✗ Scaling
✗ Management
```

None of the required information was found.

### Context Precision

The retrieved information is completely unrelated to the question.

```text
Relevant Chunks: 0%
Irrelevant Chunks: 100%
```

Therefore:

```text
Context Precision → Low
```

### Context Recall

None of the facts required by the reference answer were retrieved.

```text
✗ Open-source
✗ Container orchestration
✗ Deployment
✗ Scaling
✗ Management
```

Therefore:

```text
Context Recall → Low
```

### Expected Outcome

| Metric | Expected |
|----------|----------|
| Context Precision | Low |
| Context Recall | Low |

---

# Scenario C: Incomplete Retrieval

## Retrieved Context

```text
Kubernetes is an open-source container orchestration platform.
```

## Analysis

The retrieved context contains:

```text
✓ Open-source
✓ Container orchestration
```

But is missing:

```text
✗ Deployment
✗ Scaling
✗ Management
```

### Context Precision

Everything that was retrieved is relevant.

No irrelevant information exists.

```text
Relevant Chunks: High
Irrelevant Chunks: None
```

Therefore:

```text
Context Precision → High
```

### Context Recall

Although the retrieved chunk is relevant, it does not contain all information necessary to answer the question completely.

Missing facts:

```text
✗ Deployment
✗ Scaling
✗ Management
```

Therefore:

```text
Context Recall → Low
```

### Key Lesson

This scenario demonstrates an important distinction:

```text
Relevant Retrieval
does not necessarily mean
Complete Retrieval
```

A retriever may return only useful information while still failing to retrieve everything needed.

### Expected Outcome

| Metric | Expected |
|----------|----------|
| Context Precision | High |
| Context Recall | Low |

---

# Scenario D: Mixed Retrieval

## Retrieved Context

```text
The Eiffel Tower is located in Paris.

Kubernetes is an open-source container orchestration platform.

It automates deployment, scaling, and management of
containerized applications.

Bananas are rich in potassium.
```

In [1]:
import os
from dotenv import load_dotenv
from datasets import Dataset
from pandas import DataFrame

# Updated Ragas metric imports
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas import evaluate

# LangChain OpenAI integrations
from ragas.llms import LangchainLLMWrapper
# Use Groq for the Judge and HuggingFace for fast, local embeddings
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

In [3]:
# 1. Load the environment variables from your .env file
load_dotenv()

# Verify the key is loaded (don't print the actual key in output!)
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY not found. Please check your .env file.")

# 2. Initialize Models
# Requires GROQ_API_KEY in your .env file
judge_model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print(f"✅ Judge LLM configured as: {judge_model.model_name}")
print(f"✅ Embeddings configured as: {embeddings.model_name}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Judge LLM configured as: llama-3.1-8b-instant
✅ Embeddings configured as: sentence-transformers/all-MiniLM-L6-v2


In [17]:
dataset_good = Dataset.from_dict({
    "question": [
        "What is Kubernetes?"
    ],

    "contexts": [[
        "Kubernetes is an open-source container orchestration platform.",
        "It automates deployment, scaling, and management of containerized applications."
    ]],

    "answer": [
        "Kubernetes is an open-source platform for automating deployment, scaling, and management of containerized applications."
    ],

    "reference": [
        "Kubernetes is an open-source container orchestration platform that automates deployment, scaling, and management of containerized applications."
    ]
})

dataset_wrong = Dataset.from_dict({
    "question": [
        "What is Kubernetes?"
    ],

    "contexts": [[
        "Tomatoes grow well in summer.",
        "Bananas are rich in potassium."
    ]],

    "answer": [
        "Kubernetes is a container orchestration platform."
    ],

    "reference": [
        "Kubernetes is an open-source container orchestration platform that automates deployment, scaling, and management of containerized applications."
    ]
})

dataset_incomplete = Dataset.from_dict({
    "question": [
        "What is Kubernetes?"
    ],

    "contexts": [[
        "Kubernetes is an open-source container orchestration platform."
    ]],

    "answer": [
        "Kubernetes automates deployment, scaling, and management of containerized applications."
    ]
    ,

    "reference": [
        "Kubernetes is an open-source container orchestration platform that automates deployment, scaling, and management of containerized applications."
    ]
})

dataset_mixed = Dataset.from_dict({
    "question": [
        "What is Kubernetes?"
    ],

    "contexts": [[
        "The Eiffel Tower is located in Paris.",
        "Bananas are rich in potassium."
        "Kubernetes is an open-source container orchestration platform.",
        "It automates deployment, scaling, and management of containerized applications.",
    ]],

    "answer": [
        "Kubernetes is an open-source container orchestration platform that automates deployment, scaling, and management of containerized applications."
    ],

    "reference": [
        "Kubernetes is an open-source container orchestration platform that automates deployment, scaling, and management of containerized applications."
    ]
})

display(DataFrame(dataset_good))
display(DataFrame(dataset_wrong))
display(DataFrame(dataset_incomplete))
display(DataFrame(dataset_mixed))

,question,contexts,answer,reference
0,What is Kubernetes?,[Kubernetes is an open-source container orches...,Kubernetes is an open-source platform for auto...,Kubernetes is an open-source container orchest...


,question,contexts,answer,reference
0,What is Kubernetes?,"[Tomatoes grow well in summer., Bananas are ri...",Kubernetes is a container orchestration platform.,Kubernetes is an open-source container orchest...


,question,contexts,answer,reference
0,What is Kubernetes?,[Kubernetes is an open-source container orches...,"Kubernetes automates deployment, scaling, and ...",Kubernetes is an open-source container orchest...


,question,contexts,answer,reference
0,What is Kubernetes?,"[The Eiffel Tower is located in Paris., Banana...",Kubernetes is an open-source container orchest...,Kubernetes is an open-source container orchest...


In [8]:
def evaluate_with_dataset(dataset, ds_name):
    print("Evaluating model outputs for dataset: "+ ds_name)

    # Execute Ragas using the cloud models
    score = evaluate(
        dataset=dataset,
        metrics=[
            faithfulness,
            answer_relevancy,
            context_precision,
            context_recall
        ],
        llm=judge_model,
        embeddings=embeddings,
        raise_exceptions=False # Prevents the entire run from failing if one evaluation errors out
    )

    # Output as a clean Pandas DataFrame for analysis
    df_results = score.to_pandas()
    display(DataFrame(df_results))
    print()

In [18]:
evaluate_with_dataset(dataset_good, 'dataset_good')
evaluate_with_dataset(dataset_wrong, 'dataset_wrong')
evaluate_with_dataset(dataset_incomplete, 'dataset_incomplete')
evaluate_with_dataset(dataset_mixed, 'dataset_mixed')

Evaluating model outputs for dataset: dataset_good


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is Kubernetes?,[Kubernetes is an open-source container orches...,Kubernetes is an open-source platform for auto...,Kubernetes is an open-source container orchest...,1.0,0.919421,1.0,0.666667



Evaluating model outputs for dataset: dataset_wrong


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is Kubernetes?,[Kubernetes is an open-source container orches...,"Kubernetes automates deployment, scaling, and ...",Kubernetes is an open-source container orchest...,1.0,0.801378,1.0,0.5



Evaluating model outputs for dataset: dataset_mixed


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is Kubernetes?,"[The Eiffel Tower is located in Paris., Banana...",Kubernetes is an open-source container orchest...,Kubernetes is an open-source container orchest...,1.0,1.0,0.333333,1.0


## Analysis

The retrieved context contains both relevant and irrelevant information.

Relevant information:

```text
✓ Open-source
✓ Container orchestration
✓ Deployment
✓ Scaling
✓ Management
```

Irrelevant information:

```text
✗ Eiffel Tower
✗ Bananas
```

### Context Recall

Recall only asks:

> Were all required facts retrieved?

The answer is:

```text
✓ Open-source
✓ Container orchestration
✓ Deployment
✓ Scaling
✓ Management
```

All required facts are present.

Therefore:

```text
Context Recall → High
```

### Why Recall Remains High

A common misunderstanding is:

> The answer did not use the Eiffel Tower and Banana chunks, so recall should decrease.

This is incorrect.

Context Recall does not evaluate whether every retrieved chunk was used.

Instead, it evaluates whether all information needed to produce the reference answer was retrieved.

The presence of extra information does not reduce recall.

### Context Precision

Precision asks:

> How much of the retrieved information was useful?

Retrieved:

```text
✓ Kubernetes definition
✓ Deployment
✓ Scaling
✓ Management

✗ Eiffel Tower
✗ Bananas
```

Since some retrieved chunks are irrelevant:

```text
Context Precision → Lower
```

### Important Observation About Ordering

RAGAS Context Precision is not simply:

```text
Relevant Chunks / Total Chunks
```

The metric also considers the ranking quality of retrieved chunks.

For example:

**Version A**

```text
1. Kubernetes definition
2. Deployment/Scaling/Management
3. Eiffel Tower
4. Bananas
```

Typically produces:

```text
High Precision
```

because relevant chunks appear first.

**Version B**

```text
1. Eiffel Tower
2. Kubernetes definition
3. Deployment/Scaling/Management
4. Bananas
```

Typically produces:

```text
Lower Precision
```

because irrelevant information appears before relevant information.

This mirrors real-world search engines, where ranking quality is important.

### Expected Outcome

| Metric | Expected |
|----------|----------|
| Context Precision | Lower |
| Context Recall | High |

---

# Visual Summary

## Scenario A – Good Retrieval

```text
Required Facts Retrieved:
✓ ✓ ✓ ✓ ✓

Noise:
None
```

Result:

```text
Precision → High
Recall → High
```

---

## Scenario B – Wrong Retrieval

```text
Required Facts Retrieved:
✗ ✗ ✗ ✗ ✗

Noise:
High
```

Result:

```text
Precision → Low
Recall → Low
```

---

## Scenario C – Incomplete Retrieval

```text
Required Facts Retrieved:
✓ ✓ ✗ ✗ ✗

Noise:
None
```

Result:

```text
Precision → High
Recall → Low
```

---

## Scenario D – Mixed Retrieval

```text
Required Facts Retrieved:
✓ ✓ ✓ ✓ ✓

Noise:
Present
```

Result:

```text
Precision → Lower
Recall → High
```

---

# Final Summary Table

| Scenario | Retrieval Quality | Context Precision | Context Recall | Explanation |
|-----------|------------------|------------------|----------------|-------------|
| A | Good | High | High | Relevant information retrieved and nothing important is missing |
| B | Wrong | Low | Low | Retrieved information is unrelated to the question |
| C | Incomplete | High | Low | Retrieved information is relevant but important facts are missing |
| D | Mixed | Lower | High | All required facts retrieved, but irrelevant information is also present |

---

# Core Takeaway

Think of the two metrics as answering different questions.

### Context Precision

```text
Of everything retrieved,
how much was useful?
```

### Context Recall

```text
Of everything needed,
how much was retrieved?
```

This distinction explains all four scenarios:

```text
Missing information
        ↓
Hurts Recall

Irrelevant information
        ↓
Hurts Precision
```

A strong retriever must achieve both:

```text
High Precision
+
High Recall
```

meaning it retrieves **only useful information** while also retrieving **all necessary information**.